In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Iterator, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd

import xgboost as xgb
import lightgbm as lgb
from lightgbm import LGBMClassifier
import shap

from src.data_prepatation import create_batched_splits

In [ ]:
from config import dir_config
compiled_dir = dir_config.data.compiled
processed_dir = dir_config.data.processed

In [ ]:
def lgb_cuda_available():
    try:
        import lightgbm as lgb
        data = lgb.Dataset([[1,2],[3,4]], label=[0,1])
        lgb.train({"device": "cuda", "verbose": -1}, data, num_boost_round=1)
        return True
    except Exception:
        return False

print(lgb_cuda_available())

In [ ]:

sleep_stage_cols = ['sleep_stage', 'sleep_stage_transition', 'sleep_stage_trans_prop']
apnea_cols = ['apnea_obstructive', 'apnea_central', 'apnea_hypopnea', 'apnea_mixed']
epoch_cols = ['epoch_id', 'epoch_start', 'epoch_end']
subject_col = []
additional_cols = ['ahi', 'oahi', 'arousal_index']

feature_max_lag = 10
target_type = 'apnea'
target_future_step = 5


USE_GPU = lgb_cuda_available()
random_seed = 2542

# Load data

In [ ]:
# get all parquet files in the processed directory
parquet_files = list(Path(processed_dir).glob('*_with_metadata.parquet'))

# remove "agg_data_with_metadata.parquet" from the list of parquet files
parquet_files = [f for f in parquet_files if f.name != "agg_data_with_metadata.parquet"]

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, classification_report, confusion_matrix

def evaluate_model(model, X, y, threshold=0.5):
    y_pred_proba = model.predict_proba(X)[:, 1]
    y_pred = (y_pred_proba >= threshold).astype(int)
    auc_roc = roc_auc_score(y, y_pred_proba)
    auc_pr = average_precision_score(y, y_pred_proba)
    f1 = f1_score(y, y_pred)
    #
    classification = classification_report(y, y_pred)
    confusion = confusion_matrix(y, y_pred)
    return {'auc_roc': auc_roc, 'auc_pr': auc_pr, 'f1': f1, 'classification_report': classification, 'confusion_matrix': confusion}

## Chunk in 0.5hr

In [ ]:
train_X, train_y, val_X, val_y, test_X, test_y, left_out = create_batched_splits(
    parquet_files,
    batch_size=360,
    gap_size=6,
    top_features=None,
    top_features_lag=0,
    target_type='apnea',
    target_future_steps=target_future_step,
    val_ratio=0.2,
    test_ratio=0.2,
    n_leave_out=5,
    random_seed=random_seed,
)

Remove range and slope as they do not contribute any additional information to model

## Check for colinearity and adjust features

In [ ]:
from collections import defaultdict
from sklearn.feature_selection import mutual_info_classif

def remove_highly_correlated_features(train_X, train_y=None, threshold=0.8, feature_weights=None):
    """
    Remove highly correlated features using a SHAP/MI-weighted maximum independent set heuristic.

    Weight priority:
      1. explicit feature_weights dict
      2. mutual information with train_y (if provided)
      3. equal weights (fallback — least preferred, causes blind dropping)

    Returns
    -------
    kept    : list of feature names to retain
    dropped : list of feature names to remove
    """
    corr_matrix = train_X.corr().abs()
    pairs = [
        (train_X.columns[i], train_X.columns[j], corr_matrix.iloc[i, j])
        for i, j in zip(*np.where(corr_matrix > threshold))
        if i < j
    ]

    if not pairs:
        print("No highly correlated pairs found.")
        return sorted(train_X.columns.tolist()), []

    all_features = set()
    adj = defaultdict(set)
    for f1, f2, _ in pairs:
        adj[f1].add(f2)
        adj[f2].add(f1)
        all_features.update([f1, f2])

    # Build weights
    if feature_weights is not None:
        weights = {f: feature_weights.get(f, 1.0) for f in all_features}
    elif train_y is not None:
        mi = mutual_info_classif(
            train_X[sorted(all_features)].fillna(0),
            train_y,
            random_state=42,
        )
        raw = dict(zip(sorted(all_features), mi))
        max_mi = max(raw.values()) or 1.0
        weights = {f: raw[f] / max_mi + 1e-6 for f in all_features}  # +eps avoids zero weight
        print("Using mutual information weights for feature selection.")
    else:
        weights = {f: 1.0 for f in all_features}
        print("Warning: no feature weights provided. Falling back to equal weights — consider passing train_y.")

    # Greedy weighted maximum independent set
    remaining = set(all_features)
    kept = set()
    while remaining:
        best = max(remaining, key=lambda f: weights[f] / (1.0 + len(adj[f] & remaining)))
        kept.add(best)
        remaining.discard(best)
        remaining -= adj[best]

    dropped = sorted(all_features - kept)
    kept = sorted(kept)

    print(f"\nKept {len(kept)} / dropped {len(dropped)} from the correlated set:")
    for f in dropped:
        neighbors = sorted(adj[f] & set(kept))
        print(f"  drop '{f}' (MI weight={weights[f]:.4f}) — kept instead: {neighbors}")

    return kept, dropped

In [ ]:
columns_to_drop = ['subject_id', 'chunk_id']
train_X = train_X.drop(columns=columns_to_drop)
val_X = val_X.drop(columns=columns_to_drop)
test_X = test_X.drop(columns=columns_to_drop)

kept, dropped = remove_highly_correlated_features(train_X, train_y, threshold=0.8)
train_X = train_X.drop(columns=dropped)
val_X = val_X.drop(columns=dropped)
test_X = test_X.drop(columns=dropped)

In [ ]:
train_X.shape, val_X.shape, test_X.shape

In [ ]:
import optuna
from sklearn.metrics import average_precision_score

def lgb_hyperparameter_fitting(train_X, train_y, val_X, val_y):
    optuna.logging.set_verbosity(optuna.logging.WARNING)


    neg = (train_y == 0).sum()
    pos = (train_y == 1).sum()
    spw = neg / pos

    BASE_PARAMS = {
        "n_estimators": 5000,
        "scale_pos_weight": spw,
        "random_state": 42,
        "metric": "average_precision",
        "verbosity": -1,
        "n_jobs": 1,                          # avoid thread contention during parallel trials
        "device": "cuda" if USE_GPU else "cpu",
    }

    FIT_CALLBACKS = [
        lgb.early_stopping(50, first_metric_only=True, verbose=False),
        lgb.log_evaluation(-1),
    ]


    def objective(trial):
        params = BASE_PARAMS | {
            "learning_rate":      trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
            "max_depth":          trial.suggest_int("max_depth", 3, 7),
            "num_leaves":         trial.suggest_int("num_leaves", 15, 63),
            "min_child_samples":  trial.suggest_int("min_child_samples", 20, 100),
            "colsample_bytree":   trial.suggest_float("colsample_bytree", 0.3, 0.8),
            "subsample":          trial.suggest_float("subsample", 0.4, 0.9),
            "reg_alpha":          trial.suggest_float("reg_alpha", 0.0, 2.0),
            "reg_lambda":         trial.suggest_float("reg_lambda", 0.0, 4.0),
        }

        model = LGBMClassifier(**params)
        model.fit(
            train_X, train_y,
            eval_set=[(val_X, val_y)],
            callbacks=FIT_CALLBACKS,
        )

        val_proba = model.predict_proba(val_X)[:, 1]
        return average_precision_score(val_y, val_proba)


    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=random_seed))
    study.optimize(
        objective,
        n_trials=50,
        n_jobs=-1,              # ← parallel trials across CPU cores,
        show_progress_bar=True,
    )

    print(f"\nBest val AUC-PR: {study.best_value:.4f}")
    print(f"Best params:     {study.best_params}")

    # --- Retrain best model, use val only for early stopping ---
    best_model_params = BASE_PARAMS | study.best_params

    model_lgb_tuned = LGBMClassifier(**best_model_params)
    model_lgb_tuned.fit(
        train_X, train_y,
        eval_set=[(val_X, val_y)],
        callbacks=FIT_CALLBACKS,
    )

    return model_lgb_tuned


### LightGBM

In [ ]:
nolag_model = lgb_hyperparameter_fitting(train_X, train_y, val_X, val_y)

print("\n--- Tuned model evaluation ---")
for split_name, X, y in [
    ("Train", train_X, train_y),
    ("Val",   val_X,   val_y),
]:
    m = evaluate_model(nolag_model, X, y)
    print(f"{split_name:5s}  AUC-ROC={m['auc_roc']:.3f}  AUC-PR={m['auc_pr']:.3f}  F1={m['f1']:.3f}")

In [ ]:
explainer = shap.TreeExplainer(nolag_model)

# --- Validation set SHAP: for plotting/evaluation ---
val_shap_values = explainer.shap_values(val_X)
val_sv = val_shap_values[1] if isinstance(val_shap_values, list) else val_shap_values
shap.summary_plot(val_sv, val_X, max_display=20)

# --- Train set SHAP: for feature selection ---
# Using train set here to select features for the lagged model;
# val/test sets are kept unseen for downstream evaluation.
train_shap_values = explainer.shap_values(train_X)
train_sv = train_shap_values[1] if isinstance(train_shap_values, list) else train_shap_values

shap_importance = pd.Series(
    np.abs(train_sv).mean(axis=0),
    index=train_X.columns,
).sort_values(ascending=False)

# Filter out subject metadata columns from feature candidates
subject_metadata = pd.read_csv(Path(processed_dir) / 'participant_info.csv')
metadata_cols = set(subject_metadata.columns)

top_shap_features = [
    col for col in shap_importance.nlargest(20).index
    if col not in metadata_cols
]

dropped = [col for col in shap_importance.nlargest(20).index if col in metadata_cols]
if dropped:
    print(f"Dropped metadata columns from top features: {dropped}")

print(f"Top SHAP features: {top_shap_features}")

### Adding lag features for top SHAP features

In [ ]:
lagged5_train_X, future0_train_y, lagged5_val_X, future0_val_y, lagged5_test_X, future0_test_y, left_out = create_batched_splits(
    parquet_files,
    batch_size=360,
    gap_size=6,
    top_features=top_shap_features,
    top_features_lag=feature_max_lag,
    target_type='apnea',
    target_future_steps=target_future_step,
    val_ratio=0.2,
    test_ratio=0.2,
    n_leave_out=5,
    random_seed=2542,
)

In [ ]:
columns_to_drop = ['subject_id', 'chunk_id']
if any(col in columns_to_drop for col in lagged5_train_X.columns):
    lagged5_train_X = lagged5_train_X.drop(columns=columns_to_drop)
    lagged5_val_X = lagged5_val_X.drop(columns=columns_to_drop)
    lagged5_test_X = lagged5_test_X.drop(columns=columns_to_drop)

In [ ]:
lagged_model = lgb_hyperparameter_fitting(lagged5_train_X, future0_train_y, lagged5_val_X, future0_val_y)

print("\n--- Tuned model evaluation ---")
for split_name, X, y in [
    ("Train", lagged5_train_X, future0_train_y),
    ("Val",   lagged5_val_X,   future0_val_y),
]:
    m = evaluate_model(lagged_model, X, y)
    print(f"{split_name:5s}  AUC-ROC={m['auc_roc']:.3f}  AUC-PR={m['auc_pr']:.3f}  F1={m['f1']:.3f}")